In [2]:
import pandas as pd
import shutil
import os
import glob
from tqdm import tqdm  # 진행바 표시 (없으면 'pip install tqdm' 하세요)

# =========================================================
# [설정] 님 컴퓨터 경로 (그대로 넣었습니다)
# =========================================================
# 1. 원본 데이터가 숨어있는 폴더 (재귀적으로 찾습니다)
source_root = r"C:\Users\home\Desktop\AI_Study\YOLO_Dataset_New_Smart_Strict_7787(2)"

# 2. 아까 받은 오답 리스트 CSV 경로
csv_path = r"C:\Users\home\Desktop\AI_Study\my_error_note(2).csv"

# 3. 오답 파일들을 모아둘 새로운 폴더 (자동 생성됨)
target_dir = r"C:\Users\home\Desktop\AI_Study\Review_Errors_Collection(2)"
# =========================================================

def collect_error_files():
    # 1. CSV 읽기
    if not os.path.exists(csv_path):
        print(f"❌ CSV 파일이 없습니다: {csv_path}")
        return
    
    try:
        df = pd.read_csv(csv_path)
        print(f"📄 오답 리스트 로드 완료: 총 {len(df)}개")
    except Exception as e:
        print(f"❌ CSV 읽기 실패 (인코딩 문제일 수 있음): {e}")
        return

    # 2. 저장할 폴더 만들기
    os.makedirs(target_dir, exist_ok=True)
    images_dir = os.path.join(target_dir, "images")
    labels_dir = os.path.join(target_dir, "labels")
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    print(f"🔍 '{source_root}' 안에서 원본 파일 위치를 찾는 중... (시간 좀 걸림)")
    
    # 3. 원본 폴더 뒤져서 파일 위치 맵핑 (파일명 -> 전체경로)
    # (폴더 구조가 복잡해도 파일명만 같으면 찾아냅니다)
    file_map = {}
    label_map = {}
    
    for root, dirs, files in os.walk(source_root):
        for file in files:
            if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                file_map[file] = os.path.join(root, file)
            elif file.endswith('.txt'):
                label_map[file] = os.path.join(root, file)

    print(f"✅ 파일 인덱싱 완료! (이미지 {len(file_map)}개 발견)")

    # 4. 복사 시작
    success_count = 0
    missing_count = 0

    print("🚀 오답 파일 수집 및 이름 변경 시작...")
    
    # tqdm은 진행바입니다. 없으면 지워도 됩니다.
    for index, row in tqdm(df.iterrows(), total=len(df)):
        file_name = row['파일명']
        gt = row['실제정답']
        pred = row['모델예측']
        conf = row['확신도(%)']

        # 원본 이미지 찾기
        if file_name in file_map:
            src_img_path = file_map[file_name]
            
            # [중요] 파일명 변경: [실제정답_예측값_확신도] 원본파일명
            # 이렇게 해야 폴더에서 봤을 때 뭐가 문제인지 바로 보임
            new_filename = f"[GT{gt}_Pred{pred}_{conf}%]_{file_name}"
            
            dst_img_path = os.path.join(images_dir, new_filename)
            
            # 이미지 복사
            shutil.copy2(src_img_path, dst_img_path)

            # 라벨 파일도 있으면 같이 복사 (파일명 맞춰서)
            txt_filename = os.path.splitext(file_name)[0] + ".txt"
            if txt_filename in label_map:
                src_txt_path = label_map[txt_filename]
                dst_txt_path = os.path.join(labels_dir, os.path.splitext(new_filename)[0] + ".txt")
                shutil.copy2(src_txt_path, dst_txt_path)
            
            success_count += 1
        else:
            missing_count += 1
            # print(f"⚠️ 원본 못 찾음: {file_name}")

    print("="*50)
    print(f"🔥 작업 끝!")
    print(f"✅ 성공: {success_count}개")
    print(f"❌ 실패(못찾음): {missing_count}개")
    print(f"📂 저장 위치: {target_dir}")
    print("   👉 폴더에 들어가서 이미지들을 훑어보세요.")
    print("="*50)

if __name__ == "__main__":
    collect_error_files()

📄 오답 리스트 로드 완료: 총 1262개
🔍 'C:\Users\home\Desktop\AI_Study\YOLO_Dataset_New_Smart_Strict_7787(2)' 안에서 원본 파일 위치를 찾는 중... (시간 좀 걸림)
✅ 파일 인덱싱 완료! (이미지 8012개 발견)
🚀 오답 파일 수집 및 이름 변경 시작...


100%|██████████| 1262/1262 [00:11<00:00, 106.23it/s]

🔥 작업 끝!
✅ 성공: 1262개
❌ 실패(못찾음): 0개
📂 저장 위치: C:\Users\home\Desktop\AI_Study\Review_Errors_Collection(2)
   👉 폴더에 들어가서 이미지들을 훑어보세요.
